# 03b — Transfer Learning: FINAL TRAINING (local GPU)

## Waste Classification AI — Phase 4

> **Run this notebook locally with a CUDA GPU** (developed/tested on an NVIDIA RTX 4050 via
> a dedicated project `.venv`). It trains all three transfer-learning models —
> **MobileNetV2**, **EfficientNet-B0**, and **ResNet50** — using the two-stage strategy from
> [`src/models/transfer.py`](../src/models/transfer.py):
>
> - **Stage A (feature extraction):** freeze the pretrained backbone, train only the new
>   classifier head, higher learning rate (`STAGE_A_LR = 1e-3`).
> - **Stage B (fine-tuning):** unfreeze the backbone's top layers, continue training the
>   whole unfrozen set at a much lower learning rate (`STAGE_B_LR = 1e-5`) — see the
>   module docstring for why fine-tuning needs a smaller LR (avoiding "catastrophic
>   forgetting" of the pretrained features).
>
> This mirrors [`03a_transfer_learning_smoke_test.ipynb`](03a_transfer_learning_smoke_test.ipynb),
> which only verified the mechanics (weights load, forward/backward run, freeze/unfreeze
> works, checkpointing works) on a tiny data slice with **zero real training** — that
> notebook's numbers are not performance metrics. This notebook produces the official
> Phase 4 results used in the Phase 6 comparison table, using the **same dataset split,
> preprocessing, and evaluation protocol** as the from-scratch CNN (Phase 3) so the
> comparison is fair.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Optional Colab fallback (not required -- this project runs locally on a CUDA GPU by default).
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/waste-classification-ai")  # <-- adjust if needed
    get_ipython().system(f"pip install -q -r {PROJECT_ROOT}/requirements.txt")

sys.path.insert(0, str(PROJECT_ROOT))
print(f"IN_COLAB = {IN_COLAB}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


In [ ]:
import json
import time

import torch
import torch.nn as nn

from src.data.dataset import WasteDataset
from src.data.download import download_trashnet
from src.data.splitter import create_stratified_split, load_split_manifest, save_split_manifest
from src.data.transforms import get_eval_transforms, get_train_transforms
from src.evaluation.metrics import (
    get_classification_report,
    get_classification_report_dict,
    get_predictions,
    plot_confusion_matrix,
    plot_training_curves,
)
from src.models.transfer import ARCHITECTURES, build_transfer_model, count_parameters, freeze_backbone, unfreeze_top_layers
from src.training.class_weights import compute_class_weights
from src.training.trainer import fit, save_history
from src.utils.config import (
    BATCH_SIZE,
    CLASS_NAMES,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_SEED,
    RESULTS_DIR,
    SPLITS_MANIFEST_PATH,
    STAGE_A_LR,
    STAGE_B_LR,
)
from src.utils.device import get_device
from src.utils.seed import set_seed

set_seed(RANDOM_SEED)
device = get_device()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}  |  CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. This will be very slow on CPU.")

download_trashnet()
if not SPLITS_MANIFEST_PATH.exists():
    manifest = create_stratified_split()
    save_split_manifest(manifest)
manifest = load_split_manifest()

train_ds = WasteDataset(split="train", transform=get_train_transforms(), manifest=manifest)
val_ds = WasteDataset(split="val", transform=get_eval_transforms(), manifest=manifest)
test_ds = WasteDataset(split="test", transform=get_eval_transforms(), manifest=manifest)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_weights = compute_class_weights(manifest, split="train").to(device)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## Per-model training config

Same class-weighted `CrossEntropyLoss` as Phase 3 (addresses the 4.3x class imbalance from
the Phase 1 EDA). `num_unfreeze_blocks` picks how many backbone blocks Stage B unfreezes —
for MobileNetV2/EfficientNet-B0 this counts from the end of their flat `features` list; for
ResNet50 it counts residual stages from the end (`1` = just `layer4`).


In [ ]:
STAGE_A_EPOCHS = 10
STAGE_B_EPOCHS = 15
PATIENCE = 5

NUM_UNFREEZE_BLOCKS = {"mobilenet_v2": 2, "efficientnet_b0": 2, "resnet50": 1}

print(f"STAGE_A_EPOCHS={STAGE_A_EPOCHS}  STAGE_B_EPOCHS={STAGE_B_EPOCHS}  PATIENCE={PATIENCE}")
print(f"STAGE_A_LR={STAGE_A_LR}  STAGE_B_LR={STAGE_B_LR}")


## Training loop — runs Stage A then Stage B for each architecture

In [ ]:
import os

all_results = {}
all_histories = {}
oom_errors = {}

for architecture in ARCHITECTURES:
    print(f"\n{'='*70}\n{architecture}\n{'='*70}")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    model = None
    try:
        model = build_transfer_model(architecture, num_classes=len(CLASS_NAMES), pretrained=True).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        checkpoint_path = MODELS_DIR / f"{architecture}_best.pt"

        # --- Stage A: feature extraction (frozen backbone, train classifier head only) ---
        freeze_backbone(model, architecture)
        stage_a_param_counts = count_parameters(model)
        print(f"[Stage A] {stage_a_param_counts}")
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=STAGE_A_LR)

        t0 = time.time()
        history_a = fit(
            model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, device=device,
            epochs=STAGE_A_EPOCHS, checkpoint_path=checkpoint_path, patience=PATIENCE,
        )
        stage_a_time = time.time() - t0
        stage_a_best_val_loss = min(history_a["val_loss"])
        stage_a_best_epoch = history_a["val_loss"].index(stage_a_best_val_loss) + 1
        stage_a_best_val_acc = history_a["val_acc"][stage_a_best_epoch - 1]

        # --- Stage B: fine-tuning (unfreeze top layers, lower LR) ---
        # Reload Stage A's best weights before continuing, so Stage B starts from the best
        # feature-extraction checkpoint rather than whatever the last Stage A epoch left behind.
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        unfreeze_top_layers(model, architecture, num_blocks=NUM_UNFREEZE_BLOCKS[architecture])
        stage_b_param_counts = count_parameters(model)
        print(f"[Stage B] {stage_b_param_counts}")
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=STAGE_B_LR)

        t0 = time.time()
        history_b = fit(
            model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, device=device,
            epochs=STAGE_B_EPOCHS, checkpoint_path=checkpoint_path, patience=PATIENCE,
        )
        stage_b_time = time.time() - t0

        # Combined history (Stage A epochs followed by Stage B epochs) for a single training-curve plot
        combined_history = {key: history_a[key] + history_b[key] for key in history_a}
        stage_b_start_epoch = len(history_a["train_loss"])
        best_epoch_combined = combined_history["val_loss"].index(min(combined_history["val_loss"])) + 1
        all_histories[architecture] = {"combined": combined_history, "stage_b_start_epoch": stage_b_start_epoch}
        save_history(combined_history, RESULTS_DIR / f"{architecture}_training_history.json")

        fig = plot_training_curves(
            combined_history, title_prefix=f"{architecture} (Stage A -> Stage B) -- ",
            save_path=FIGURES_DIR / f"{architecture}_training_curves.png",
        )

        # --- Evaluate best checkpoint (across both stages) on the test set ---
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        model.to(device)
        y_true, y_pred, y_probs = get_predictions(model, test_loader, device)
        test_accuracy = (y_true == y_pred).mean()
        report_dict = get_classification_report_dict(y_true, y_pred, CLASS_NAMES)

        print(f"\n{architecture} test accuracy: {test_accuracy:.4f}")
        print(get_classification_report(y_true, y_pred, CLASS_NAMES))

        plot_confusion_matrix(
            y_true, y_pred, CLASS_NAMES, normalize=True,
            title=f"{architecture} -- Normalized Confusion Matrix (Test Set)",
            save_path=FIGURES_DIR / f"{architecture}_confusion_matrix.png",
        )

        model_size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)
        peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else None
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"

        results = {
            "model_name": architecture,
            "oom_error": False,
            "stage_a_epochs_run": len(history_a["train_loss"]),
            "stage_a_best_epoch": stage_a_best_epoch,
            "stage_a_best_val_loss": stage_a_best_val_loss,
            "stage_a_best_val_acc": stage_a_best_val_acc,
            "stage_a_time_sec": stage_a_time,
            "stage_b_epochs_run": len(history_b["train_loss"]),
            "stage_b_time_sec": stage_b_time,
            "best_epoch_combined": best_epoch_combined,
            "total_training_time_sec": stage_a_time + stage_b_time,
            "test_accuracy": float(test_accuracy),
            "test_precision_macro": report_dict["macro avg"]["precision"],
            "test_recall_macro": report_dict["macro avg"]["recall"],
            "test_f1_macro": report_dict["macro avg"]["f1-score"],
            "test_precision_weighted": report_dict["weighted avg"]["precision"],
            "test_recall_weighted": report_dict["weighted avg"]["recall"],
            "test_f1_weighted": report_dict["weighted avg"]["f1-score"],
            "per_class": {name: report_dict[name] for name in CLASS_NAMES},
            "total_params": stage_b_param_counts["total_params"],
            "trainable_params_stage_a": stage_a_param_counts["trainable_params"],
            "trainable_params_stage_b": stage_b_param_counts["trainable_params"],
            "model_size_mb": model_size_mb,
            "peak_gpu_memory_mb": peak_gpu_mb,
            "gpu_used": gpu_name,
            "random_seed": RANDOM_SEED,
        }
        with open(RESULTS_DIR / f"{architecture}_test_metrics.json", "w") as f:
            json.dump(results, f, indent=2)
        all_results[architecture] = results

    except torch.OutOfMemoryError as e:
        print(f"\n!!! CUDA OUT OF MEMORY while training {architecture}: {e}")
        oom_errors[architecture] = str(e)
        all_results[architecture] = {"model_name": architecture, "oom_error": True, "error_message": str(e)}

    finally:
        # Free GPU memory before moving to the next model -- models run strictly
        # sequentially (one architecture fully finishes, including evaluation, before
        # the next one is even constructed) so GPU memory never has to hold two models.
        if model is not None:
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n\nAll architectures processed.")
if oom_errors:
    print("CUDA/OOM ISSUES OCCURRED:", oom_errors)
else:
    print("No CUDA/OOM issues occurred for any model.")


## Summary across all three transfer-learning models

In [ ]:
import pandas as pd

# Official CNN baseline (Phase 3, approved -- values fixed, never recomputed here)
CNN_BASELINE = {
    "model": "CNN (baseline)",
    "test_accuracy": 0.6368,
    "test_precision_macro": 0.665,
    "test_recall_macro": 0.658,
    "test_f1_macro": 0.621,
    "test_f1_weighted": 0.650,
    "total_params": 390918,
    "total_training_time_sec": 796,
}

rows = [CNN_BASELINE]
for name, r in all_results.items():
    if r.get("oom_error"):
        rows.append({"model": name, "test_accuracy": None, "test_precision_macro": None,
                      "test_recall_macro": None, "test_f1_macro": None, "test_f1_weighted": None,
                      "total_params": None, "total_training_time_sec": None})
        continue
    rows.append({
        "model": name,
        "test_accuracy": r["test_accuracy"],
        "test_precision_macro": r["test_precision_macro"],
        "test_recall_macro": r["test_recall_macro"],
        "test_f1_macro": r["test_f1_macro"],
        "test_f1_weighted": r["test_f1_weighted"],
        "total_params": r["total_params"],
        "total_training_time_sec": r["total_training_time_sec"],
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULTS_DIR / "phase4_model_comparison.csv", index=False)
summary_df


**Interpretation:** *(fill in after this actually runs -- compare accuracy/F1 against
the Phase 3 CNN-from-scratch baseline (63.68% accuracy, 0.621 macro F1, 390,918 params,
~796s), note which architecture performed best and whether that lines up with expectations
from parameter count / architecture design, and describe each model's Stage A -> Stage B
convergence pattern. Do NOT pick a winner on accuracy alone -- weigh F1, inference speed,
parameter count, and model size, per the project's model-selection requirement (Phase 13).)*
